[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-01-intro-mlflow.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Introduction to MLflow: Tracking Your First Experiment
**certified-journeys / mlflow-certified** · Day 1 · Getting Started

> **Goal for today:** Install MLflow, train a simple sklearn model, and log params, metrics, and an artifact so you can see the run appear in the MLflow UI.

In [ ]:
%pip install -q mlflow scikit-learn pandas matplotlib

## Step 1 · What is MLflow?

MLflow is an open-source platform with four core components:

| Component | Purpose |
|---|---|
| **Tracking** | Log params, metrics, artifacts, and code versions |
| **Projects** | Package ML code for reproducible runs |
| **Models** | Standard format to save and deploy models |
| **Registry** | Central model store with versioning and stage transitions |

Today we focus entirely on **Tracking** — the component you will use every single day.

The tracking API revolves around three primitives:
- **Param** — a key-value configuration that does not change during a run (e.g. `learning_rate=0.01`)
- **Metric** — a numeric value you can log at each step (e.g. accuracy at epoch 5)
- **Artifact** — any file you want to store alongside the run (model weights, plots, CSVs)

In [ ]:
import mlflow

# Print the MLflow version to confirm the install
print("MLflow version:", mlflow.__version__)

# By default MLflow stores runs in ./mlruns in the current directory.
# In Colab this is ephemeral, but the API is identical to a local or remote server.
print("Default tracking URI:", mlflow.get_tracking_uri())

### What just happened?
- **`mlflow.__version__`** confirms the package is installed correctly.
- **`get_tracking_uri()`** shows where run data will be written — `mlruns/` by default.
- Every run you create will be stored under this URI; changing it redirects all logging to a remote server.
- No server process is needed for local tracking — MLflow writes directly to the filesystem.

## Step 2 · Your First Run: `log_param` and `log_metric`

Every call to `mlflow.log_param` or `mlflow.log_metric` must happen **inside an active run**.

The simplest way is the `with mlflow.start_run():` context manager — it opens the run on entry and closes it on exit, even if an exception is raised.

```python
with mlflow.start_run():
    mlflow.log_param("key", value)   # any JSON-serialisable value
    mlflow.log_metric("key", number) # must be a float
```

Use `log_params({...})` and `log_metrics({...})` to batch-log multiple values in one call.

In [ ]:
import mlflow
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the Iris dataset — a classic 3-class classification problem
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- MLflow run starts here ---
with mlflow.start_run(run_name="logistic-regression-baseline") as run:

    # 1. Log hyperparameters
    params = {"C": 1.0, "max_iter": 200, "solver": "lbfgs"}
    mlflow.log_params(params)  # batch version — one round-trip to the store

    # 2. Train the model
    clf = LogisticRegression(**params)
    clf.fit(X_train, y_train)

    # 3. Evaluate and log metrics
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test))
    mlflow.log_metrics({"train_accuracy": train_acc, "test_accuracy": test_acc})

    print(f"Run ID : {run.info.run_id}")
    print(f"Train  : {train_acc:.4f}")
    print(f"Test   : {test_acc:.4f}")

### What just happened?
- **`mlflow.start_run(run_name=...)`** creates a new run entry; the `as run` alias exposes `run.info.run_id`.
- **`log_params`** stores the hyperparameter dict — values are converted to strings internally.
- **`log_metrics`** stores floating-point values; these are what you see in the Metrics tab of the UI.
- **The `with` block auto-closes the run** — no risk of leaving an orphaned active run behind.
- Check `mlruns/` in your file browser; a new directory appeared with the run's metadata.

## Step 3 · Logging an Artifact

An **artifact** is any local file you copy into the run's artifact store.

```python
mlflow.log_artifact(local_path)           # single file
mlflow.log_artifact(local_path, artifact_path="subdir")  # into a subdirectory
mlflow.log_artifacts(local_dir)           # entire directory tree
```

Common artifacts to log:
- A **confusion matrix PNG** so you can visually audit predictions
- The **feature column list** as a text file — critical for reproducing inference
- A **requirements.txt** snapshot so you know exactly what was installed

> `log_artifact` **copies** the file — the original on disk is not moved or deleted.

In [ ]:
import os
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for Colab and scripts
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Re-train (same params) so this cell is self-contained
clf2 = LogisticRegression(C=1.0, max_iter=200, solver="lbfgs")
clf2.fit(X_train, y_train)

# --- Build a confusion matrix PNG ---
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(clf2, X_test, y_test, ax=ax)
ax.set_title("Iris — Logistic Regression")
fig.tight_layout()

plot_path = "/tmp/confusion_matrix.png"
fig.savefig(plot_path, dpi=100)
plt.close(fig)   # free memory

# --- Write a tiny feature-names file ---
feature_path = "/tmp/feature_names.txt"
iris = load_iris()
with open(feature_path, "w") as f:
    f.write("\n".join(iris.feature_names))

# --- Log both artifacts inside a new run ---
with mlflow.start_run(run_name="logistic-regression-with-artifacts") as run:
    mlflow.log_params({"C": 1.0, "max_iter": 200, "solver": "lbfgs"})
    acc = accuracy_score(y_test, clf2.predict(X_test))
    mlflow.log_metric("test_accuracy", acc)

    # log_artifact copies the file into the run's artifact store
    mlflow.log_artifact(plot_path, artifact_path="plots")
    mlflow.log_artifact(feature_path, artifact_path="metadata")

    print(f"Run ID       : {run.info.run_id}")
    print(f"Test accuracy: {acc:.4f}")
    print(f"Artifacts logged: confusion_matrix.png, feature_names.txt")

### What just happened?
- **`artifact_path`** creates a subdirectory inside the run's artifact root — keeps plots, models, and metadata organised.
- The PNG was generated with **matplotlib's `Agg` backend**, which works headlessly in scripts and CI.
- **Artifacts are immutable once logged** — if you need a v2, start a new run or a new artifact path.
- In the MLflow UI, click the run → Artifacts tab to browse all logged files and preview images inline.

## Step 4 · Inspecting Runs Programmatically

You do not need the UI to inspect runs. The **MlflowClient** API lets you query run data in notebooks or scripts:

```python
from mlflow.tracking import MlflowClient
client = MlflowClient()

client.search_runs(experiment_ids=["0"])  # returns list of Run objects
run_data = client.get_run(run_id)         # fetch a specific run
run_data.data.params                      # dict of params
run_data.data.metrics                     # dict of metrics (last logged value)
```

This is useful for post-run analysis, reporting scripts, and automated model selection.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

client = MlflowClient()

# Experiment "0" is the default experiment created automatically
runs = client.search_runs(
    experiment_ids=["0"],
    order_by=["metrics.test_accuracy DESC"]
)

# Build a summary DataFrame for easy inspection
rows = []
for r in runs:
    rows.append({
        "run_id"    : r.info.run_id[:8],   # truncate for readability
        "run_name"  : r.info.run_name,
        "status"    : r.info.status,
        "C"         : r.data.params.get("C"),
        "test_acc"  : r.data.metrics.get("test_accuracy"),
        "train_acc" : r.data.metrics.get("train_accuracy"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

### What just happened?
- **`search_runs`** returns a list of `Run` objects ordered by whatever metric you choose.
- **`r.data.params`** and **`r.data.metrics`** are plain Python dicts — easy to put into a DataFrame.
- The `order_by` clause uses the same `metrics.<name>` / `params.<name>` syntax as the UI filter bar.
- **`status`** will be `FINISHED`, `RUNNING`, or `FAILED` — useful for debugging crashed runs.

## Step 5 · Running the MLflow UI (local only)

In Colab the UI is not accessible from outside the VM, but on your local machine you launch it with:

```bash
mlflow ui
# Opens http://127.0.0.1:5000 in your browser
```

Things to explore in the UI:

| Section | What you see |
|---|---|
| **Experiments list** | All named experiments and run counts |
| **Run list** | Table of runs with params, metrics, and tags |
| **Run detail → Overview** | Git commit, source file, start time |
| **Run detail → Metrics** | Time-series chart (useful for epoch-by-epoch logging) |
| **Run detail → Artifacts** | File browser; images render inline |

The cell below prints a summary equivalent to what the UI would show, so you can verify everything was logged correctly without leaving the notebook.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Fetch all runs from the default experiment
runs = client.search_runs(experiment_ids=["0"])

for r in runs:
    print("=" * 60)
    print(f"Run      : {r.info.run_name} ({r.info.run_id[:8]})")
    print(f"Status   : {r.info.status}")
    print(f"Params   : {r.data.params}")
    print(f"Metrics  : {r.data.metrics}")

    # List artifacts if any were logged
    artifacts = client.list_artifacts(r.info.run_id)
    if artifacts:
        print("Artifacts:")
        for a in artifacts:
            print(f"  {'[dir]' if a.is_dir else '     '} {a.path}")
        # Also list files inside subdirectories
        for a in artifacts:
            if a.is_dir:
                for sub in client.list_artifacts(r.info.run_id, a.path):
                    print(f"         {a.path}/{sub.path}")
    print()

### What just happened?
- **`client.list_artifacts(run_id)`** returns top-level entries; pass an `artifact_path` to recurse.
- **`a.is_dir`** distinguishes directories (like `plots/`) from leaf files.
- This programmatic view mirrors what you see in the UI's Artifacts browser.
- **On your local machine**, run `mlflow ui` now and open http://127.0.0.1:5000 to see the same data visually.

In [ ]:
# Challenge: Train a second model variant and compare it with the baseline
#
# Instructions:
#   1. Create a new MLflow run named "logistic-regression-high-C"
#   2. Train a LogisticRegression with C=10.0 (everything else the same)
#   3. Log the params and test_accuracy metric
#   4. Log a confusion matrix PNG as an artifact under "plots/"
#   5. Use MlflowClient.search_runs to print both runs side-by-side
#
# Scaffold — fill in the blanks:

# with mlflow.start_run(run_name=???) as run:
#     params_high_c = {"C": ???, "max_iter": 200, "solver": "lbfgs"}
#     mlflow.log_params(???)
#     clf_high = LogisticRegression(**params_high_c)
#     clf_high.fit(X_train, y_train)
#     acc = accuracy_score(y_test, clf_high.predict(X_test))
#     mlflow.log_metric(???, ???)
#     # generate + log confusion matrix PNG here
#     pass

# Your solution here


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow.start_run()` | Opens a run context; always use `with` to auto-close it |
| `log_param` / `log_params` | Store hyperparameters as key-value strings |
| `log_metric` / `log_metrics` | Store numeric evaluation results |
| `log_artifact` | Copy a local file into the run's artifact store |
| `MlflowClient.search_runs` | Query runs programmatically without the UI |
| `artifact_path` | Optional subdirectory inside the run's artifact root |
| Default experiment | Runs without `set_experiment` go to experiment `"0"` |

> **Tip:** The MLflow UI is the fastest way to understand what was logged — always run `mlflow ui` in a terminal while experimenting.

---
## What's next
**Day 2** → Learn to create named experiments, start and end runs explicitly, log time-series metrics across epochs, and compare runs side-by-side in the UI.

Mark Day 1 complete in your [tracker](../index.html).